## Primitive Logic --> Stateless Logic Circuits

- We've already see one level of upward abstraction in the previous section
    - PMOS/NMOS are just primitive on/off switches, but they combine to give you logical expressions
    - Logical expressions can be combined to give you novel logical behaviour (NOR + NAND + AND gives you XOR)

- Now, let's see how these primitive logic gates can get us to stateless operations! We will go through the implementations for the core primitives:
    - Adders
    - Multiplexers
    - Decoders
    - Comparators
    - Barrel Shifters & Rotators

- These primitives will give us some useful conceptual extensions, which we will also look at:
    - Subtractors
    - Multipliers
    - Encoders & Priority Encoders

- Finally, we will see how the combinations of these will jointly make up the Arithmetic Logic Unit (ALU) of a computer

### Adders

- Adders are the basic building blocks of all computer arithmetic. By chaining primitive logic gates together, we can translate Boolean logic directly into binary addition!

- There are 3 types of adders we will study
    - Half Adder
        - A half adder performs a sum of 2 bits
        - It outputs a sum bit and a carry bit
        - We consider it a half adder because it only outputs a carry bit, but doesn't accept a carry bit
    - Full Adder
        - A full adder performs the sum of 2 bits, AND a carry bit
        - It outputs a sum bit and a carry bit
    - Ripple Adder
        - This is a composite adder, which connects $N$ Full Adders in series to add $N$-bit numbers        
        - The carry output from each bit position "ripples" into the carry input of the next higher bit position

        

In [ ]:
from utils import *

def half_adder(a: TRANSISTOR_OUTPUT, b: TRANSISTOR_OUTPUT) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
            A ───┬─────────┐
                 │  ┌───┐  ├─── [cmos_XOR] ─── Sum
            B ───┼──┤XOR│──┘
                 │  └───┘
                 │  ┌───┐
                 └──┤AND│────── [cmos_AND] ─── Carry
                    └───┘
    '''
    ## - If (0, 0) --> (sum: 0, carry: 0)
    ##    - XOR(0,0) = 0, AND(0,0) = 0
    ## - If (0, 1) --> (sum: 1, carry: 0)
    ##    - XOR(0,1) = 1, AND(0,1) = 0
    ## - If (1, 0) --> (sum: 1, carry: 0)
    ##    - XOR(1,0) = 1, AND(0,1) = 0
    ## - If (1, 1) --> (sum: 0, carry: 1)
    ##    - XOR(1,1) = 0, AND(0,1) = 1

    # Returns 1 if a != b, else 0
    sum_out = cmos_XOR(a, b)

    # Returns 1 if a == b == 1, else 0
    carry_out = cmos_AND(a, b)

    return sum_out, carry_out


def full_adder(a: TRANSISTOR_OUTPUT, b: TRANSISTOR_OUTPUT, c_in: TRANSISTOR_OUTPUT = GROUND) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
        A, B ──────> [Half Adder 1] ─── (Sum1, Carry1)
                           │
        Sum1, C_in ─> [Half Adder 2] ─── (Final Sum, Carry2)
                           │
        Carry1, Carry2 ─> [cmos_OR] ─── Final Carry Out

    Note that the maximum a full adder will be asked to add is 1+1+1, which means it will never exceed 3.
    This is why the output of a full adder can simply be a sum, carry
    The sum can only be 0 or 1, since it is the sum of 2 binary values
    The carry can represent a maximum of the 2^1 (since it is a carry, it represents the binary value for the 
    next significant position)

    - (0, 0, 0) -> (0, 0)
        - HA(a=0, b=0) = s1=0, c1=0
        - HA(s1=0, c=0) = s2=0, c2=0
        - OR(c1=0, c=0) = c3=0
        - Output(s2=0, c3=0)
    - (0, 0, 1) | (0, 1, 0) | (1, 0, 0) -> (1, 0)
        - HA(a=1, b=0) = s1=1, c1=0
        - HA(s1=1, c=0) = s2=1, c2=0
        - OR(c1=0, c2=0) = c3=0
        - Output(s2=1, c3=0)
    - (0, 1, 1) | (1, 1, 0) | (1, 0, 1) -> (0, 1)
        - HA(a=1, b=1) = s1=0, c1=1
        - HA(s1=0, c=0) = s2=0, c2=0
        - OR(c1=1, c2=0) = c3=1
        - Output(s2=0, c3=1)
    - (1, 1, 1) -> (1, 1)
        - HA(a=1, b=1) = s1=0, c1=1
        - HA(s1=0, c=1) = s2=1, c2=0
        - OR(c1=1, c2=0) = c3=1
        - Output(s2=1, c3=1)
    '''
    
    
    sum1, carry1 = half_adder(a, b)

    final_sum, carry2 = half_adder(sum1, c_in)
    
    final_carry = cmos_OR(carry1, carry2)
    return final_sum, final_carry


def ripple_carry_adder(a_bits: list[TRANSISTOR_OUTPUT], b_bits: list[TRANSISTOR_OUTPUT]) -> tuple[list[TRANSISTOR_OUTPUT], TRANSISTOR_OUTPUT]:
    '''
    Ripple-Carry Adder processing LSB to MSB.
    Takes two equal-length lists of bit signals (ordered LSB -> MSB).
    Returns (sum_bits, final_carry_out)

        LSB (Bit 0)                 Bit 1                    MSB (Bit 2)
        a_bits[0] = 1            a_bits[1] = 1              a_bits[2] = 1
        b_bits[0] = 1            b_bits[1] = 1              b_bits[2] = 1
                │                        │                          │
                │   ┌─────────┐          │   ┌─────────┐            │   ┌─────────┐
                ├───┤         │          ├───┤         │            ├───┤         │
                │   │ Full    │          │   │ Full    │            │   │ Full    │
                └───┤ Adder 0 │          └───┤ Adder 1 │            └───┤ Adder 2 │
                    │         │              │         │                │         │
    GROUND ─────────┤ c_in    │  ┌───────────┤ c_in    │    ┌───────────┤ c_in    │
    (c_in = 0)      │         │  │ (c_out=1) │         │    │ (c_out=1) │         │
                    │   c_out ├──┘           │   c_out ├────┘           │   c_out ├─── Final Carry
                    │   sum   ├──┐           │   sum   ├──┐             │   sum   ├──┐ (carry = 1)
                    └─────────┘  │           └─────────┘  │             └─────────┘  │
                                ▼                        ▼                          ▼
                            sum_bits[0]              sum_bits[1]                sum_bits[2]
                            (Sum = 0)                (Sum = 1)                  (Sum = 1)

    
    ============
       EXAMPLE
    ============
    - a_bits = 5 = [1,0,1]
    - b_bits = 3 = [0,1,1]
    - carry = 0
    - sum_bits = []
    
    - Going from LSB to MSB:
        - full_adder(a=1, b=1, c=0) 
            - s1=0, c1=1
            - sum_bits = [,0]
        - full_adder(a=0, b=1, c=c1=1)
            - s2=0, c2=1
            - sum_bits = [,0,0]
        - full_adder(a=1, b=0, c=c2=1)
            - s3=0, c3=1
            - sum_bits = [0,0,0]
    
    - Final value:
        - Concatenate sum_bits and prepend the final carry
        - output = 1000 = 8 = 5 + 3

    '''
    carry: TRANSISTOR_OUTPUT = GROUND
    sum_bits: list[TRANSISTOR_OUTPUT] = []
    
    for bit_a, bit_b in zip(a_bits, b_bits):
        s, carry = full_adder(bit_a, bit_b, carry)
        sum_bits.append(s)
        
    return sum_bits, carry

### Multiplexers

- A Multiplexer, or MUX, is simply a "router". It takes in some **data inputs**, and depending on some **control inputs** (also known as **selector lines**), it returns 1 of them 

- The variations of multiplexers basically differ in terms of how many inputs it can choose between
    - The more inputs it needs to choose from, the more **control inputs** it must take in
    - For example, a 2-to-1 MUX has to select between 2 data inputs, and this can be done using 1 control input (since 1 input has 2 binary states).
    - Whereas a 4-to-1 MUX requires 2 control inputs (since 2 bits can represent 4 states)

In [ ]:
from utils import *


def mux_2to1(d0: TRANSISTOR_OUTPUT, d1: TRANSISTOR_OUTPUT, sel: TRANSISTOR_OUTPUT) -> TRANSISTOR_OUTPUT:
    '''
    2-to-1 MUX.
    Selects d0 when sel is GROUND (0), and d1 when sel is POWER (1).

            D0 ───┬─────────┐
                  │  ┌───┐  ├─── [cmos_AND] ─── Path0 ──┐
        ~Sel ─────┼──┤AND│──┘                           │
                  │  └───┘                              ├─── [cmos_OR] ─── Output
            D1 ───┼─────────┐                           │
                  │  ┌───┐  ├─── [cmos_AND] ─── Path1 ──┘
         Sel ─────┼──┤AND│──┘
                     └───┘
    
    Logic Truth Table:
    - If sel == 0:
        - ~sel = 1
        - path0 = AND(d0, ~sel)
            - If D0 = 0 => Output = 0
            - If D0 = 1 => Output = 1
        - path1 = AND(d1, sel)
            - If D1 = 0 => Output = 0
            - If D1 = 1 => Output = 0
        - Output = OR(path0, path1) = 1
    '''
    # Invert the selector bit to enable the d0 branch
    not_sel = cmos_NOT(sel)

    # Route d0 if sel is 0, route d1 if sel is 1
    path0 = cmos_AND(d0, not_sel)
    path1 = cmos_AND(d1, sel)

    # Combine the active path (only one path can be active at a time)
    return cmos_OR(path0, path1)


def mux_4to1(
    d0: TRANSISTOR_OUTPUT, 
    d1: TRANSISTOR_OUTPUT, 
    d2: TRANSISTOR_OUTPUT, 
    d3: TRANSISTOR_OUTPUT, 
    sel: tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]
) -> TRANSISTOR_OUTPUT:
    '''
    4-to-1 Multiplexer constructed hierarchically using 2-to-1 MUXes.
    
    Inputs:
    - d0, d1, d2, d3: Data lines
    - sel: (sel1, sel0) 
    
    Hierarchy Diagram:
        D0 ────┐
               ├─── [ MUX Stage 1: Level 0 ] ─── Out0 ──┐
        D1 ────┘                 │                      │
                             sel[0] (LSB)               ├─── [ MUX Stage 2 ] ─── Output
        D2 ────┐                                        │             │
               ├─── [ MUX Stage 1: Level 1 ] ─── Out1 ──┘         sel[1] (MSB)
        D3 ────┘                 │
                             sel[0] (LSB)

    - sel = (0, 0) -> Out0 = d0, Out1 = d2 -> Output = d0
    - sel = (0, 1) -> Out0 = d1, Out1 = d3 -> Output = d1
    - sel = (1, 0) -> Out0 = d0, Out1 = d2 -> Output = d2
    - sel = (1, 1) -> Out0 = d1, Out1 = d3 -> Output = d3
    '''
    sel1, sel0 = sel

    # Stage 1: Use sel0 (LSB) to select between pairs (d0 vs d1) and (d2 vs d3)
    level0_out = mux_2to1(d0, d1, sel0)
    level1_out = mux_2to1(d2, d3, sel0)

    # Stage 2: Use sel1 (MSB) to select between the winning pair from Stage 1
    return mux_2to1(level0_out, level1_out, sel1)


def n_bit_mux_2to1(
    a_word: list[TRANSISTOR_OUTPUT], 
    b_word: list[TRANSISTOR_OUTPUT], 
    sel: TRANSISTOR_OUTPUT
) -> list[TRANSISTOR_OUTPUT]:
    '''
    N-bit 2-to-1 Bus Multiplexer.
    Switches an entire multi-bit word (bus) based on a single select signal.
    
    Takes two equal-length lists of bit signals (ordered LSB -> MSB).
    Returns a selected word of equal length.

        a_word[0] ───┐                                 b_word[0] ───┐
                     ├─── [ MUX 0 ] ─── out_word[0]                 ├─── [ MUX 1 ] ─── out_word[1]
        b_word[0] ───┘         │                       b_word[1] ───┘         │
                             Sel                                            Sel

    ============
       EXAMPLE
    ============
    - a_word = [1, 0, 1] (5 in decimal)
    - b_word = [0, 1, 1] (3 in decimal)
    - sel = POWER (1)
    
    - Output:
        - MUX 0: (a[0]=1, b[0]=0, sel=1) -> 0
        - MUX 1: (a[1]=0, b[1]=1, sel=1) -> 1
        - MUX 2: (a[2]=1, b[2]=1, sel=1) -> 1
    - result_word = [0, 1, 1] (Successfully selected b_word)
    '''
    selected_word: list[TRANSISTOR_OUTPUT] = []

    for bit_a, bit_b in zip(a_word, b_word):
        output_bit = mux_2to1(bit_a, bit_b, sel)
        selected_word.append(output_bit)

    return selected_word

### Decoders

- A Decoder is a combinational circuit that takes some $N$-bit binary input, and outputs exactly $2^N$ lines, one and only one of which must be activated
    - That is, a 1-to-2 decoder takes in 1 input (0, or 1), and provides 2 outputs, one of which will be 1 (01, 10)
    - Similarly, a 2-to-4 decoder takes in 2 inputs (00, 01, 10, or 11), and provides 4 outputs, one of which will be 1 (0001, 0010, 0100, 1000)

- One thing that jumps out from this: this is uncannily similar to what the Multiplexer was doing in the previous section!

- Decoders are fundamental to computer architecture
    - Memory Addressing: Selecting a specific row in RAM using a memory address
    - Instruction Decoding: Translating opcode bits into control signals for specific hardware units

In [ ]:
from utils import *


def decoder_1to2(a0: TRANSISTOR_OUTPUT) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
    1-to-2 Decoder.
    Takes 1 input bit (A0) and Outputs 2 lines (Y0, Y1). Either Y0=1 or Y1=1

            A0 ───┬─────────┐
                  │  ┌───┐  └─── [cmos_AND] ─── Y1 (Active when A0=1)
            1 ────┼──┤AND│
                  │  └───┘
                  │  ┌───┐  ┌─── [cmos_NOT] ──┐
                  └──┤AND│──┤                 ├─── Y0 (Active when A0=0)
                     └───┘  └─────────────────┘

    ============
       EXAMPLE
    ============
    - Input: a0 = 0
        - y0 = cmos_NOT(a0) = 1
        - y1 = a0 = 0
        - Returns (y0, y1) = (1, 0)
    - Input: a0 = 1
        - y0 = cmos_NOT(a0) = 0
        - y1 = 1
        - Returns (y0, y1) = (0, 1)
    '''
    y0 = cmos_NOT(a0)
    y1 = a0
    return y0, y1


def decoder_2to4(
    a1: TRANSISTOR_OUTPUT, 
    a0: TRANSISTOR_OUTPUT
) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
    2-to-4 Decoder.
    Takes 2 input bits (A0, A1) and Outputs 4 lines (Y0, Y1, Y2, Y3). Either Y0=1, Y1=1, Y2=1, Y3=1

        A1 ────┬──────────────┐
               │   ┌─────┐    │
        A0 ────┼───┤ AND ├────┼───────────────> Y0 (00)
               │   └─────┘    │
               │              │   ┌─────┐
               ├──────────────┼───┤ AND ├─────> Y1 (01)
               │              │   └─────┘
               │  (Inv Gates) │   ┌─────┐
               └──────────────┼───┤ AND ├─────> Y2 (10)
                              │   └─────┘
                              │   ┌─────┐
                              └───┤ AND ├─────> Y3 (11)
                                  └─────┘
    ============
       EXAMPLE
    ============
    - Basically we iterate through every combination of 2 bits (00, 01, 10, 11)
    - To do this, we have (a0, a1, ~a0, ~a1)
    - Then we simply assign each of outputs y0,y1,y2,y3 to one of the combinations
        - (a0,a1) --> y0
        - (a0,~a1) --> y1
        - (~a0,a1) --> y2
        - (~a0,~a1) --> y3
    - No matter the combination of a0 and a1, only 1 case will every return 1 by definition
        - Because AND requires both values to be 1, and the NOT guarantees that can only happen in 1 case
    '''
    not_a1 = cmos_NOT(a1)
    not_a0 = cmos_NOT(a0)

    # Calculate active-high lines via minterm AND logic
    y0 = cmos_AND(not_a1, not_a0)
    y1 = cmos_AND(not_a1, a0)
    y2 = cmos_AND(a1, not_a0)
    y3 = cmos_AND(a1, a0)

    return y0, y1, y2, y3


def decoder_3to8(
    a2: TRANSISTOR_OUTPUT, 
    a1: TRANSISTOR_OUTPUT, 
    a0: TRANSISTOR_OUTPUT
) -> tuple[
    TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT,
    TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT
]:
    '''
    3-to-8 Decoder constructed using 2-to-4 logic

                (A1, A0) ───────> [ 2-to-4 Decoder ] ─── (Sub0, Sub1, Sub2, Sub3)
                                        │
                ┌──────────────────────────┴──────────────────────────┐
                ▼                                                     ▼
        A2=0 Branch (Lower)                                  A2=1 Branch (Upper)
        (Y0 = Sub0 AND ~A2)                                  (Y4 = Sub0 AND A2)
        (Y1 = Sub1 AND ~A2)                                  (Y5 = Sub1 AND A2)
        (Y2 = Sub2 AND ~A2)                                  (Y6 = Sub2 AND A2)
        (Y3 = Sub3 AND ~A2)                                  (Y7 = Sub3 AND A2)

    - For 3 inputs, we have 8 possibilities (000, 001, 010, 100, 011, 101, 110, 111)
    - We borrow the same approach from the decoder_2to4, but apply it sequentially instead
    - Idea is: 
        - We first decide which combination of a0 and a1 we are dealing with
            - So decoder_2to4(a0, a1)
            - This gives us (sub0, sub1, sub2, sub3), which can be one of (0001, 0010, 0100, 1000)
        - Then depending on which of these we get, we combine this bit-wise with a2=0 and ~a2=1
            - Suppose we have list[0,0,0,1]
                - list[0,0,0,1] AND a2=0 --> [y0=0, y1=0, y2=0, y3=0]
                - list[0,0,0,1] AND ~a2=1 --> [y4=0, y5=0, y6=0, y7=1]
        - Remember, this works because the output from  `decoder_2to4` is guaranteed to have one and only 1 positive bit 
        - AND a2 is guaranteed to be 0 or 1
    '''
    # Step 1: Decode the two lower bits (A1, A0) into 4 intermediate lines
    sub0, sub1, sub2, sub3 = decoder_2to4(a0, a1)
    not_a2 = cmos_NOT(a2)

    # Step 2: Route to lower 4 outputs (Active when A2 is 0)
    y0 = cmos_AND(sub0, not_a2)
    y1 = cmos_AND(sub1, not_a2)
    y2 = cmos_AND(sub2, not_a2)
    y3 = cmos_AND(sub3, not_a2)

    # Step 3: Route to upper 4 outputs (Active when A2 is 1)
    y4 = cmos_AND(sub0, a2)
    y5 = cmos_AND(sub1, a2)
    y6 = cmos_AND(sub2, a2)
    y7 = cmos_AND(sub3, a2)

    return y0, y1, y2, y3, y4, y5, y6, y7

### Comparators

- A comparator is what is used to evaluate the relative value of 2 binary values
    - Depending on the relative values of A and B, a comparator's output flags will tell you if A > B, A == B, or A < B

- The most straightforward application of Comparators is simply for `if ()` conditions 

- How to check for equality? We'll start with the 1 bit case, then generalise to N bits

In [ ]:
from utils import *


def comparator_1bit(a: TRANSISTOR_OUTPUT, b: TRANSISTOR_OUTPUT) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
    1-Bit Magnitude Comparator
    Given two 1-bit inputs (A, B), return a tuple of 1-bit values representing (a_gt_b, a_eq_b, a_lt_b)

            A ───┬──────────────┐
                 │   ┌──────┐   ├─── [cmos_AND] ─── A_GT_B (A=1, B=0)
            B ───┼───┤~B    ├───┘
                 │   └──────┘
                 │   ┌──────┐
                 ├───┤ cmos ├─── A_EQ_B (A == B)
                 │   │ XNOR │
            B ───┼───┤      ├───
                 │   └──────┘
                 │   ┌──────┐
            ~A ──┼───┤ cmos ├─── A_LT_B (A=0, B=1)
            B ───┴───┤ AND  ├───
                     └──────┘
    
    - Intuition
        - a_gt_b = cmos_AND(a, not_b)
            - This works, because if the inversion of B is equal to A, and A = 1, then B must have been 0 to begin with
            - If A were 0, then the AND will return 0 regardless, because 0 cannot be more than anything in binary
        - xor_out = cmos_XOR(a, b), a_eq_b = cmos_NOT(xor_out)
            - This checks because XOR returns 1 if either A or B were 1, but not if they are both 1
            - So if XOR returns 1, they cannot be equal. 
            - Hence, A == B is just the NOT of the XOR
            - This is also known as XNOR
        - a_lt_b = cmos_AND(not_a, b)
            - Same logic as a_gt_b, but in reverse
    
    - (A=0, B=0), (A=1, B=1) => (0, 1, 0)
        - We'll work through the (0,0) case, but the (1,1) case is symmetric
        - ~A = 1, ~B = 1
        - Check if A > B
            - AND(A, ~B) = 0 
        - Check if A == B
            - x1 = XOR(A, B) = 0
            - NOT(x1) = 1
        - Check if A < B
            - AND(~A, B) = 0
    - (A=1, B=0) => (1, 0, 0)
        - ~A = 0, ~B = 1
        - Check if A > B
            - AND(A, ~B) = 1 
        - Check if A == B
            - x1 = XOR(A, B) = 1
            - NOT(x1) = 0
        - Check if A < B
            - AND(~A, B) = 0
    - (A=0, B=1) => (0, 0, 1)
        - ~A = 1, ~B = 0
        - Check if A > B
            - AND(A, ~B) = 0 
        - Check if A == B
            - x1 = XOR(A, B) = 1
            - NOT(x1) = 0
        - Check if A < B
            - AND(~A, B) = 1
    '''
    not_a = cmos_NOT(a)
    not_b = cmos_NOT(b)

    # A > B: A is 1 AND B is 0
    a_gt_b = cmos_AND(a, not_b)

    # A == B: Both are 0 or both are 1 (XNOR)
    xor_out = cmos_XOR(a, b)
    a_eq_b = cmos_NOT(xor_out)

    # A < B: A is 0 AND B is 1
    a_lt_b = cmos_AND(not_a, b)

    return a_gt_b, a_eq_b, a_lt_b

def comparator_nbit(
    a_bits: list[TRANSISTOR_OUTPUT], 
    b_bits: list[TRANSISTOR_OUTPUT]
) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
    N-Bit Magnitude Comparator 
    
    - Same as comparator_1bit, returns (a_gt_b, a_eq_b, a_lt_b)
    - When we have multiple bits, we go from the most significant bit (MSB) to least significant bit (LSB)
    - Why?
        - Because for any binary number, the magnitude of MSB strictly dominates the combination of all the less significant bits
        - e.g. 10 > 01, 100 > 011, 1000 > 0111, holds for binary numbers for any size
        - So if MSB_A > MSB_B, it must be true that A > B    
    '''

    # Process from MSB to LSB 
    for bit_a, bit_b in zip(a_bits, b_bits):
        gt, eq, lt = comparator_1bit(bit_a, bit_b)

        if gt:
            return (POWER, GROUND, GROUND)
        elif lt:
            return (GROUND, GROUND, POWER)
        else:
            continue

    return GROUND, POWER, GROUND

### Barrel Shifters / Rotators

- Think of a Barrel shifter/rotator as a multi-position bit swapper. Basically this is what enables bit shifting; when you write `1 << 2` in Python, this single logical processor is what handles the "left shift"

- There are 2 ways to shift a binary number
    - Logical Shift: After shifting, any number that falls off the end of the shift direction gets chopped off, and the new number coming in is 0
        - Suppose we have 1001, and we want to do a right shift by 2
        - Then we have _ _ 1 0, with the "01" falling over the edge
        - And we fill the blanks with 0, giving us 0010
    - Rotational Shift: After shifting, if a number falls off the end, loop it back to the start
        - Suppose we have 1001, and we want to do a right shift by 2
        - This gives us 0110
        - 1001 >> 1 pushes 1 off the right edge and onto the left, so we have 1100
        - Then 1100 >> 1 pushes 0 off the right edge, so we have 0110

In [ ]:
def barrel_rotator_right_4bit(
    data: list[TRANSISTOR_OUTPUT], 
    shift_amt: tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]
) -> list[TRANSISTOR_OUTPUT]:
    '''
    4-Bit Barrel Rotator Right (ROR).
    
    Inputs:
        - 4-bit data list [D0, D1, D2, D3] (MSB -> LSB)
        - 2-bit shift amount tuple [S1, S2] (because 2 bits has 4 combinations)
    
        Returns a rotated 4-bit word

            Data [D0, D1, D2, D3]
                    │
                    ▼
        ┌─────────────────────────┐
        │Stage 0: MUX Bank (1-bit)│ <── S0
        └────────────┬────────────┘
                    │ Intermediate [I0, I1, I2, I3]
                    ▼
        ┌─────────────────────────┐
        │Stage 1: MUX Bank (2-bit)│ <── S1
        └────────────┬────────────┘
                    ▼
        Output [O0, O1, O2, O3]

    - Recall that a 1-bit MUX takes in 2 data inputs, and returns either of them depending on a the control input flag

    - So how do we perform a rotation?
        - Suppose we have [D0,D1,D2,D3], and we want to perform a right rotation of size 3 
        - That just means we want the final state to be [D1,D2,D3,D0]

    - The trick here is entirely how we think about the inputs to `shift_amt`
        - This can be seen as a binary number, representing the number of shifts we require for each input value 
    
    - Since `shift_amt` has $N$ bits, we get from `[D0,D1,D2,D3]` => `[D1,D2,D3,D0]` in N steps!
        - For each of the N steps, we use the binary value for that particular digit of `shift_amt`
        - For example, if `shift_amt` = [1, 1], we perform the right shift in 2 steps
        - Right shift by one step, since the first value is 1, hence 1 * 2^0 --> `[D0,D1,D2,D3]` => `[D3,D0,D1,D2]`
        - Right shift by two steps, since the second value is 1, hence 1 * 2^1 --> `[D3,D0,D1,D2]` => `[D1,D2,D3,D0]`

    - The genius of this stepwise decomposition is that for N control inputs, you can produce shifts of up to 2^N 
    using just mux_2to1
        - If we wanted to do this all at once, we would have needed a mux_4to1 for control input size of N=2
    '''
    s1, s0 = shift_amt
    d0, d1, d2, d3 = data

    # ------------------------------------------------------------------
    # STAGE 0: Shift Right by 0 or 1 position 
    # If s0=0: Keep original [D0, D1, D2, D3]
    # If s0=1: Shift right by 1 -> [D3, D0, D1, D2]
    # ------------------------------------------------------------------
    i0 = mux_2to1(d0, d3, s0) # Wrap-around bit
    i1 = mux_2to1(d1, d0, s0)
    i2 = mux_2to1(d2, d1, s0)
    i3 = mux_2to1(d3, d2, s0)

    # ------------------------------------------------------------------
    # STAGE 1: Shift Right by 2 positions (if s1 is HIGH)
    # If s1=0: Keep Stage 0 output [I0, I1, I2, I3]
    # If s1=1: Shift right by 2 -> [I2, I3, I0, I1] (I0, I1 wrap)
    # ------------------------------------------------------------------
    o0 = mux_2to1(i0, i2, s1) # Wrap-around bit
    o1 = mux_2to1(i1, i3, s1) # Wrap-around bit
    o2 = mux_2to1(i2, i0, s1)
    o3 = mux_2to1(i3, i1, s1)

    return [o0, o1, o2, o3]


def barrel_logical_right_4bit(
    data: list[TRANSISTOR_OUTPUT], 
    shift_amt: tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]
) -> list[TRANSISTOR_OUTPUT]:
    '''
    4-Bit Barrel Rotator Right (ROR).
        
    Inputs:
        - 4-bit data list [D0, D1, D2, D3] (MSB -> LSB)
        - 2-bit shift amount tuple [S1, S2] (because 2 bits has 4 combinations)

    - The idea is the same as barrel_rotator, in that we are using mux_2to1 N times to perform the rotation, 
    where N=len(shift_amt)
    - However, the difference in that values that go over the edge no longer loop back around. Instead, we replace 
    it with 0

    - So Suppose we have [D0,D1,D2,D3], and we want to perform a right rotation of size 3 
        - That now means we want [0, 0, 0, D0]
        - To do this, let's shift right by 1
            - [D0,D1,D2,D3] => [0,D0,D1,D2] = [i0,i1,i2,i3]
        - Then shift right by 2
            - [0,D0,D1,D2] => [0,0,0,D0]

    '''
    s1, s0 = shift_amt
    d0, d1, d2, d3 = data

    # STAGE 0: Shift by 1 bit (Fill index 3 with GROUND)
    i0 = mux_2to1(d0, GROUND, s0)
    i1 = mux_2to1(d1, d0, s0)
    i2 = mux_2to1(d2, d1, s0)
    i3 = mux_2to1(d3, d2, s0)

    # STAGE 1: Shift by 2 bits (Fill indices 2 & 3 with GROUND)
    o0 = mux_2to1(i0, GROUND, s1)
    o1 = mux_2to1(i1, GROUND, s1)
    o2 = mux_2to1(i2, i0, s1)
    o3 = mux_2to1(i3, i1, s1)

    return [o0, o1, o2, o3]

### Subtractors

- Subtractors are the exact opposite of adders. Instead of outputting a "carry", we output a "borrow" value

- Note that no modern computers are built with subtractors anymore. Instead, modern computers use the two-complement system which lets us perform addition AND subtraction with only adders
    - See section `Using Adders as Subtractors with Two's Complement`
    - We will go through this section purely as a learning exercise

In [ ]:
from utils import *


def half_subtractor(a: TRANSISTOR_OUTPUT, b: TRANSISTOR_OUTPUT) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
    Computes (a - b) for two single bits.
    Returns (difference, borrow_out).

            A ───┬─────────┐
                 │  ┌───┐  ├─── [cmos_XOR] ─── Difference
            B ───┼──┤XOR│──┘
                 │  └───┘
                 │  ┌───┐  ┌─── [cmos_NOT] ──┐
            ~A ──┼──┤AND│──┤                 ├─── Borrow Out
            B ───┴──└───┘  └─────────────────┘

    - Intuition
        - diff = cmos_XOR(a, b)
            - The diff between A and B is either 1 or 0
            - An XOR will return 1 for (1,0) or (0,1), and 0 otherwise
            - Thus, it measures the diff
        - not_a = cmos_NOT(a=0) = 1
            - Since this does a-b, we need to distinguish between (1,0) and (0,1)
            - The XOR above returns 1 for both, but in the second case, we need to `borrow`
        - borrow = cmos_AND(not_a, b)
            - Since this is AND, the cases (0,0) and (1,0) both set borrow as 0. 
                - This makes sense; if b=0, then we never need to borrow, since we subtract 0
            - In the case (1,1), not_a is set as 0, so borrow is also 0
            - Only in the case (0,1) will we have the situation where AND(not_a, b) = 1
        - The borrow bit is borrowed from the next highest power! So in this case, it is worth 2!

 
    - (a=0, b=0) => (d=0, b=0)
        - diff = cmos_XOR(a=0, b=0) = 0
        - not_a = cmos_NOT(a=0) = 1
        - borrow = cmos_AND(not_a=1, b=0) = 0
        - Output: (diff=0, borrow=0) -> 0 - 0 doesn't need borrowing
    - (a=1, b=0) => (d=1, b=0)
        - diff = cmos_XOR(a=1, b=0) = 1
        - not_a = cmos_NOT(a=1) = 0
        - borrow = cmos_AND(not_a=0, b=0) = 0
        - Output: (diff=1, borrow=0) -> 1 - 0 doesn't need borrowing
    - (a=0, b=1) => (d=1, b=1)
        - diff = cmos_XOR(a=0, b=1) = 1
        - not_a = cmos_NOT(a=0) = 1
        - borrow = cmos_AND(not_a=1, b=1) = 1
        - Output: (diff=1, borrow=1) -> 0 - 1 needs borrowing
        - NOTE: The borrow bit is borrowed from the next highest power, so here it is read as -2+1 = -1
    - (a=1, b=1) => (d=0, b=0)
            - diff = cmos_XOR(a=1, b=1) = 0
            - not_a = cmos_NOT(a=1) = 0
            - borrow = cmos_AND(not_a=0, b=1) = 0
            - Output: (diff=0, borrow=0) -> 1 - 1 doesn't needs borrowing

    '''
    difference = cmos_XOR(a, b)
    not_a = cmos_NOT(a)
    borrow_out = cmos_AND(not_a, b)

    return difference, borrow_out


def full_subtractor(
    a: TRANSISTOR_OUTPUT, 
    b: TRANSISTOR_OUTPUT, 
    b_in: TRANSISTOR_OUTPUT = GROUND
) -> tuple[TRANSISTOR_OUTPUT, TRANSISTOR_OUTPUT]:
    '''
    Full Subtractor: Computes (a - b - b_in) considering an incoming Borrow bit

    - Intuition
        - The inputs are read as A - B - B_in
        - For subtraction, order doesn't matter, so hs(a, b) --> hs(a, b_in) is the same either way
        - Remember, every borrow is worth -2!

    - (a=0, b=0, br=0) => (d=0, b=0)
        - hs(a=0, b=0) = d1=0, br1=0
        - hs(d1=0, br=0) = d2=0, br2=0
        - OR(br1=0, br2=0) = br3 = 0
        - Output: d2=0, br3=0
    - (a=0, b=0, br=1) || (a=0, b=1, br=0) => (d=1, b=1)
        - hs(a=0, b=0) = d1=0, br1=0
        - hs(d1=0, br=1) = d2=1, br2=1
        - OR(br1=0, br2=1) = br3 = 1
        - Output: d2=1, br3=1 ==> d2 + br3 = 1 + -2 = -1
    - (a=0, b=1, br=1) => (d=1, b=1)
        - hs(a=0, b=1) = d1=1, br1=1
        - hs(d1=1, br=1) = d2=0, br2=0
        - OR(br1=1, br2=0) = br3 = 1
        - Output: d2=0, br3=1 ==> d2 + br3 = 0 + -2 = -2
    - (a=1, b=1, br=0) || (a=1, b=0, br=1) => (d=0, b=0)
        - hs(a=1, b=1) = d1=0, br1=0
        - hs(d1=0, br=0) = d2=0, br2=0
        - OR(br1=0, br2=0) = br3 = 0
        - Output: d2=0, br3=0 ==> d2 + br3 = 0 + 0 = 0
    - (a=1, b=1, br=1) => (d=1, b=1)
        - hs(a=1, b=1) = d1=0, br1=0
        - hs(d1=0, br=1) = d2=1, br2=1
        - OR(br1=0, br2=1) = br3 = 1
        - Output: d2=1, br3=1 ==> d2 + br3 = 1 + -2 = -1
    - (a=1, b=0, br=0) => (d=1, b=0)
        - hs(a=1, b=0) = d1=1, br1=0
        - hs(d1=1, br=0) = d2=1, br2=0
        - OR(br1=0, br2=0) = br3 = 0
        - Output: d2=1, br3=0 ==> d2 + br3 = 1 + 0 = 1
    '''
    diff1, borrow1 = half_subtractor(a, b)
    final_diff, borrow2 = half_subtractor(diff1, b_in)
    final_borrow = cmos_OR(borrow1, borrow2)

    return final_diff, final_borrow

### Using Adders as Subtractors with Two's Complement

- In the sections above, we discussed separate logic gates for addition and subtraction (Adders and Subtractors)

- But in every modern computer, subtractors are never used. Instead, we adopt a number system known as **Two's Complement**, which allows us to use Adders to do subtraction!

- Why do we do this? Simply, by reusing adders for subtraction, we free up space on the circuit board for other logic gates!

- How does two's complement work?
    - 2’s Complement is a mathematical system for representing signed integers (both positive and negative) using a fixed number of binary bits
    - In this system, the MSB (Most significant bit) is reserved for representing the sign of a number. And we do this simply by retaining its magnitude, but reversing its sign!
    - Example: `1101`
        - Interpreting this as a regular binary number, we have `1*8 + 1*4 + 0*2 + 1*1 = 13`
        - In two's complement, we instead treat the MSB as negative. So we have `-1*8 + 1*4 + 0*2 + 1*1 = -3`
    
- This system gives us a natural way to convert from a positive to negative number; simply start from the positive, invert all positions, and add 1!
    - Example: Suppose we want to find the binary representation of -5
    - Start from 5: `0101`
    - Invert: `1010`
    - Add 1: `1011`
    - Verifying: `-1*8 + 0*4 + 1*2 + 1*1 = -5`

- Because of this property, subtraction naturally beccomes addition! Why?
    - Suppose we have a binary value $x$
    - Suppose its inversion is $\bar{x}$
    - Let the length of x's 
    - Then by definition, $x + \bar{x} = 2$




In [ ]:
def adder_subtractor_4bit(
    a_bits: list[TRANSISTOR_OUTPUT], 
    b_bits: list[TRANSISTOR_OUTPUT], 
    sub: TRANSISTOR_OUTPUT = GROUND
) -> tuple[list[TRANSISTOR_OUTPUT], TRANSISTOR_OUTPUT]:
    '''
    Unified 4-Bit Adder / Subtractor using 2's Complement.
    - If sub = GROUND (0): Computes (A + B)
    - If sub = POWER  (1): Computes (A - B)

             A_bits        B_bits
               │             │
               │        ┌────┴────┐
               │        │ XOR Gate│ <── sub control signal
               │        └────┬────┘
               ▼             ▼
          ┌───────────────────────┐
          │  Ripple Carry Adder   │ <── c_in = sub
          └──────────┬────────────┘
                     ▼
             Result (Sum / Diff)

    ============
       EXAMPLE
    ============
    - Subtraction: 5 - 3  (a_bits=[1,0,1,0], b_bits=[1,1,0,0], sub=1)
    - Step 1: Invert b_bits with XOR(b, sub=1):
        - b_bits becomes [0, 0, 1, 1]  (1's Complement of 3)
    - Step 2: Pass into Adder with c_in = 1 (completes 2's complement):
        - 5 + (-3) = [0, 1, 0, 0] (Decimal 2!)
    '''
    # Step 1: Conditionally invert b_bits using XOR
    b_prepared: list[TRANSISTOR_OUTPUT] = []
    for bit_b in b_bits:
        inverted_or_same = cmos_XOR(bit_b, sub)
        b_prepared.append(inverted_or_same)

    # Step 2: Run through standard ripple carry adder, using sub as initial carry_in!
    result_bits: list[TRANSISTOR_OUTPUT] = []
    carry = sub

    for bit_a, bit_b in zip(a_bits, b_prepared):
        s, carry = full_adder(bit_a, bit_b, carry)
        result_bits.append(s)

    return result_bits, carry